# Создание порождающей матрицы по проверочной

In [1]:
import numpy as np

+ $H \in \{0, 1\}^{(n-k) \times n}$ - проверочная матрица, ее мы составляем сами
+ $G \in \{0, 1\}^{k \times n}$ - порождающая матрица пока, что мы ее получаем готовую

In [2]:
H = np.loadtxt('../data/matrix.txt', delimiter=',', dtype=int)
H.shape

(100, 200)

## Приведение к систематическому виду

$$
H = \left[Q^T I^{(n - k)}\right]
$$

In [3]:
class Operation:
    def __init__(self, type: str, first: int, second: int):
        self.type = type 
        self.first = first
        self.second = second 

def to_sistematic(matrix, operations):
    rows, cols = matrix.shape
    n = cols
    k = cols - rows

    for i in range(n - k):
        diag_col = i + k
        if matrix[i, diag_col] != 1:
            for j in range(n):
                if matrix[i, j] == 1:
                    operations.append(Operation('xor', diag_col, j))
                    matrix[:, diag_col] ^= matrix[:, j]
                    break
        for j_k in range(n):
            if matrix[i, j_k] == 1 and j_k != diag_col:
                operations.append(Operation('xor', j_k, diag_col))
                matrix[:, j_k] ^= matrix[:, diag_col]

operations = []
sistematic_H = H.copy()
to_sistematic(sistematic_H, operations)


## Обратное применение операций

$$
    G = \left[ I^{(k)} Q \right]
$$

In [4]:
rows, cols = H.shape
n = cols
k = cols - rows

sistematic_G = np.hstack((np.eye(k, dtype=int), np.zeros((k, n - k), dtype=int)))

for operation in operations[::-1]:
    first, second = operation.first, operation.second
    sistematic_G[:, second] ^= sistematic_G[:, first]
np.savetxt("../data/g_matrix.txt", sistematic_G, delimiter=',', fmt='%d')

def dot_for_codes(a: np.array, b: np.array) -> np.array:
    return np.array(a @ b % 2)

check = dot_for_codes(sistematic_G, H.T)
check


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])